# Notebook 02 — Preprocesamiento y Partición Temporal

## Objetivo

Implementar funciones reutilizables de limpieza de texto y generar splits temporales (70/15/15) para:
- **Subconjunto político** (experimento principal)
- **Dataset completo** (experimento complementario)

> Los modelos aprenden patrones lingüísticos del dataset, no verifican hechos.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.io import save_split
from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

FIGURE_PREFIX = '02_'

## 1. Carga y preparación inicial

In [2]:

from nlp.preprocessing import (
    add_clean_text_columns, parse_dates, temporal_split,
    class_distribution_report, filter_politics_subset, drop_content_duplicates,
)

fake_df = pd.read_csv(DATA_RAW / 'Fake.csv')
true_df = pd.read_csv(DATA_RAW / 'True.csv')
fake_df['label'] = 1
true_df['label'] = 0
df = pd.concat([fake_df, true_df], ignore_index=True)
df = parse_dates(df)
df = df.dropna(subset=['parsed_date']).reset_index(drop=True)
print(f'Registros con fecha válida: {len(df):,}')


Registros con fecha válida: 44,888


## 2. Eliminación de duplicados

En el EDA se detectaron duplicados por `title` + `text`. Se eliminan **antes del split temporal** para:
- evitar que el mismo artículo aparezca en train y test (data leakage);
- no inflar métricas por filas repetidas.

Criterio: conservar la **primera** ocurrencia de cada par `(title, text)`.

In [3]:

df, dedup_stats = drop_content_duplicates(df)
print(f"Filas antes:  {dedup_stats['rows_before']:,}")
print(f"Filas después: {dedup_stats['rows_after']:,}")
print(f"Eliminadas:    {dedup_stats['removed']:,}")
if dedup_stats['label_conflicts']:
    print(f"Grupos duplicados con etiquetas distintas (se conservó la primera): {dedup_stats['label_conflicts']}")


Filas antes:  44,888
Filas después: 39,099
Eliminadas:    5,789


## 3. Funciones de preprocesamiento de texto

In [4]:

# Demostración de las funciones de limpieza
from nlp.preprocessing import clean_text, normalize_source_markers

sample = (
    "Check this OUT!!! https://example.com/news Trump won 45% in 2016 "
    "Reuters said WHAT??? Visit www.fake.com"
)
print('Original:', sample)
print('Con stopwords:', clean_text(sample, remove_stopwords=False))
print('Sin stopwords:', clean_text(sample, remove_stopwords=True))
print('Fuentes normalizadas:', normalize_source_markers(
    clean_text(sample, remove_stopwords=False)
))


Original: Check this OUT!!! https://example.com/news Trump won 45% in 2016 Reuters said WHAT??? Visit www.fake.com
Con stopwords: check this out!!! URL trump won in reuters said what??? visit URL
Sin stopwords: check out!!! URL trump reuters said what??? visit URL
Fuentes normalizadas: check this out!!! URL trump won in [SOURCE] said what??? visit URL


### Variantes de texto generadas

Para cada uno de `title_text`, `body_text`, `full_text`:
- `clean_*_with_stopwords`
- `clean_*_without_stopwords`

Las URLs se reemplazan por `[URL]` (no se eliminan). Se preservan `!` y `?` para análisis posterior.

In [5]:

df = add_clean_text_columns(df, lemmatize=False)
politics_df = filter_politics_subset(df, include_optional=False)
print(f'Subconjunto político: {len(politics_df):,} registros')
print(politics_df.groupby('label').size())


Subconjunto político: 18,050 registros
label
0    11217
1     6833
dtype: int64


## 4. Split temporal (70% train / 15% val / 15% test)

In [6]:

def process_and_save(dataset, prefix):
    train, val, test = temporal_split(dataset)
    for name, split in [('train', train), ('val', val), ('test', test)]:
        print(f'\n=== {prefix}_{name} ===')
        print(class_distribution_report(split))
        save_split(split, prefix, name)
        print(f'Guardado: {DATA_PROCESSED / f"{prefix}_{name}"} (.parquet + .csv)')
    return train, val, test

pol_train, pol_val, pol_test = process_and_save(politics_df, 'politics')
full_train, full_val, full_test = process_and_save(df, 'full')



=== politics_train ===
      class  count  percentage  split_total
0  real (0)   7569       59.91        12635
1  fake (1)   5066       40.09        12635
Guardado: /Users/lucasrouco/workspace/ITBA/NLP/nlp/data/processed/politics_train (.parquet + .csv)

=== politics_val ===
      class  count  percentage  split_total
0  real (0)   1842       68.05         2707
1  fake (1)    865       31.95         2707
Guardado: /Users/lucasrouco/workspace/ITBA/NLP/nlp/data/processed/politics_val (.parquet + .csv)

=== politics_test ===
      class  count  percentage  split_total
0  real (0)   1806       66.69         2708
1  fake (1)    902       33.31         2708
Guardado: /Users/lucasrouco/workspace/ITBA/NLP/nlp/data/processed/politics_test (.parquet + .csv)

=== full_train ===
      class  count  percentage  split_total
0  real (0)  10860       39.68        27369
1  fake (1)  16509       60.32        27369
Guardado: /Users/lucasrouco/workspace/ITBA/NLP/nlp/data/processed/full_train (.parquet + 

### Verificación de balance en splits


In [7]:

def check_imbalance(train, val, test, threshold=0.65):
    for name, split in [('train', train), ('val', val), ('test', test)]:
        fake_pct = split['label'].mean()
        status = 'OK' if 0.35 <= fake_pct <= threshold else 'DESBALANCEADO'
        print(f'{name}: {fake_pct:.1%} fake — {status}')

print('--- Subconjunto político ---')
check_imbalance(pol_train, pol_val, pol_test)
print('\n--- Dataset completo ---')
check_imbalance(full_train, full_val, full_test)

split_rows = []
for scope, splits in [('politics', (pol_train, pol_val, pol_test)), ('full', (full_train, full_val, full_test))]:
    for split_name, split in zip(['train', 'val', 'test'], splits):
        split_rows.append({
            'scope': scope,
            'split': split_name,
            'fake_pct': split['label'].mean() * 100,
        })
split_df = pd.DataFrame(split_rows)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=split_df, x='split', y='fake_pct', hue='scope', ax=ax)
ax.set_ylabel('% fake')
ax.set_xlabel('Split')
ax.set_title('Porcentaje de fake por split temporal')
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}split_class_balance.png')
plt.show()


--- Subconjunto político ---
train: 40.1% fake — OK
val: 32.0% fake — DESBALANCEADO
test: 33.3% fake — DESBALANCEADO

--- Dataset completo ---
train: 60.3% fake — OK
val: 12.3% fake — DESBALANCEADO
test: 11.5% fake — DESBALANCEADO


## 5. Análisis de signos `!` y `?` y URLs

Evaluamos si estos patrones aparecen más en una clase que en otra (información potencial para el modelo).

In [8]:

pattern_rows = []
for label, name in [(0, 'real'), (1, 'fake')]:
    sub = politics_df[politics_df['label'] == label]
    excl_pct = sub['full_text'].str.contains('!', regex=False).mean() * 100
    quest_pct = sub['full_text'].str.contains('?', regex=False).mean() * 100
    url_pct = sub['full_text'].str.contains(r'https?://|www\.', regex=True).mean() * 100
    print(f'{name}: ! en {excl_pct:.1f}%, ? en {quest_pct:.1f}%, URLs en {url_pct:.1f}%')
    pattern_rows.extend([
        {'class': name, 'pattern': '!', 'pct': excl_pct},
        {'class': name, 'pattern': '?', 'pct': quest_pct},
        {'class': name, 'pattern': 'URL', 'pct': url_pct},
    ])
pattern_df = pd.DataFrame(pattern_rows)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pattern_df, x='pattern', y='pct', hue='class', ax=ax)
ax.set_ylabel('% de artículos')
ax.set_xlabel('Patrón')
ax.set_title('Signos de puntuación y URLs por clase (politics)')
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}punctuation_url_by_class.png')
plt.show()


real: ! en 5.3%, ? en 8.2%, URLs en 0.3%
fake: ! en 51.7%, ? en 48.1%, URLs en 13.7%


## Conclusiones

- Se generaron 6 archivos en `data/processed/`: `politics_{train,val,test}.csv` y `full_{train,val,test}.csv`.
- Cada archivo incluye columnas de texto crudo y preprocesado (con/sin stopwords).
- El split es **temporal**: train = más antiguo, test = más reciente, evitando leakage temporal.
- `subject` no se usa como feature; solo definió el filtro político.